# 07주차 실습: 파일 스트림·워터마크·복구

강의: [PDF](../pdf/week07_structured_streaming.pdf) · **PDF 실제 페이지 7–30** (인쇄 번호와 다를 수 있음)

## 학습 목표
기존 Kafka 메시지 스키마로 유한한 파일 스트림을 실행하고 체크포인트 재시작을 검증한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

## 기존 실습 연결
기본 실습 후 [W07_Spark_Structured_Streaming.ipynb](W07_Spark_Structured_Streaming.ipynb)로 이어갑니다. 호환 방법과 수정 사항은 COMPATIBILITY.md를 확인하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 입력과 워터마크 집계

In [ ]:
import uuid
from labkit.common import spark_session, event_schema, make_events
from pyspark.sql import functions as F
spark=spark_session('week07')
out=output_dir('week07')/uuid.uuid4().hex; incoming=out/'incoming'; incoming.mkdir(parents=True)
events=make_events(600)
events.iloc[:300].to_json(incoming/'batch01.json',orient='records',lines=True)
stream=spark.readStream.schema(event_schema()).json(str(incoming))
timed=stream.withColumn('event_time_ts',F.to_timestamp('event_time'))
windowed=(timed.withWatermark('event_time_ts','2 minutes')
 .groupBy(F.window('event_time_ts','5 minutes'), 'product_id').count())
def start_query():
    return (windowed.writeStream.format('parquet').outputMode('append')
     .option('path',str(out/'result')).option('checkpointLocation',str(out/'checkpoint'))
     .trigger(availableNow=True).start())
q=start_query()
try:
    if not q.awaitTermination(120): raise TimeoutError('stream timeout')
    print(q.lastProgress)
finally:
    q.stop()

## 워터마크 전진과 동일 체크포인트 재시작

In [ ]:
# 미래 이벤트를 추가해야 이전 window가 append 출력으로 확정된다.
future=make_events(1); future['event_id']='future'; future['event_time']='2026-01-01T00:20:00Z'
future.to_json(incoming/'batch02.json',orient='records',lines=True)
q=start_query()
try:
    if not q.awaitTermination(120): raise TimeoutError('stream timeout')
    save_json(out/'progress.json',q.recentProgress)
finally: q.stop()
result=spark.read.parquet(str(out/'result')); result.show(truncate=False)
before=result.agg(F.sum('count')).first()[0]
assert before == 300
q=start_query()  # 새 파일 없음: 같은 offset을 다시 출력하지 않는다.
try:
    if not q.awaitTermination(120): raise TimeoutError('stream timeout')
finally: q.stop()
assert spark.read.parquet(str(out/'result')).agg(F.sum('count')).first()[0] == before
spark.stop()

## 확장 과제 및 제출
5분 tumbling을 10분/5분 sliding으로 바꾸고 별도 체크포인트에서 실행하세요. 지연 이벤트를 후속 배치로 넣어 recentProgress의 numRowsDroppedByWatermark를 관찰하세요. 워터마크 밖 데이터는 처리될 수도 있으며 완전성 보장을 벗어납니다.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.